In [1]:
pip install opencv-python numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Lane Detection - Projeto Completo

Pipeline completo de detecção de faixas em vídeo usando OpenCV

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path

# ==================== CONFIGURACAO ====================
VIDEO_PATH = r"C:\Users\lucas\OneDrive\Documents\Lane Detection\teste.mp4"
OUTPUT_DIR = "outputs"

Path(OUTPUT_DIR).mkdir(exist_ok=True)

# ==================== UTILITARIOS ====================
def process_video(process_func, output_name):
    """Processa o video frame a frame e salva o resultado."""
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        raise ValueError(f"Nao foi possivel abrir o video: {VIDEO_PATH}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(
        f"{OUTPUT_DIR}/{output_name}.mp4",
        fourcc,
        fps,
        (width, height),
    )

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        out.write(process_func(frame))
        frame_count += 1

    cap.release()
    out.release()
    print(f"Video salvo: {OUTPUT_DIR}/{output_name}.mp4 ({frame_count} frames)")

## Etapa 1: Funções Base de Processamento

In [ ]:
def lane_roi_polygon(frame):
    """Trapezio que cobre apenas a pista visivel."""
    height, width = frame.shape[:2]
    return np.array([[
        (0, int(height * 0.68)),
        (int(width * 0.24), int(height * 0.45)),
        (int(width * 0.76), int(height * 0.45)),
        (width, int(height * 0.68)),
    ]], dtype=np.int32)


def apply_roi(frame):
    """Remove ceu, postes, fios, calcadas proximas e capo do carro."""
    mask = np.zeros_like(frame)
    fill = 255 if frame.ndim == 2 else (255,) * frame.shape[2]
    cv2.fillPoly(mask, lane_roi_polygon(frame), fill)
    return cv2.bitwise_and(frame, mask)


def build_color_mask(frame):
    """Seleciona pintura branca/cinza-clara e amarela sob iluminacao variavel."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask_white = cv2.inRange(
        hsv, np.array([0, 0, 125]), np.array([180, 105, 255])
    )
    mask_yellow = cv2.inRange(
        hsv, np.array([12, 45, 80]), np.array([42, 255, 255])
    )
    combined = cv2.bitwise_or(mask_white, mask_yellow)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)
    return apply_roi(combined)


def build_lane_edges(frame):
    """Combina bordas de cor com bordas estruturais da pista."""
    color = build_color_mask(frame)
    color_edges = cv2.Canny(cv2.GaussianBlur(color, (5, 5), 0), 25, 90)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    gray_edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 25, 85)
    return apply_roi(cv2.bitwise_or(color_edges, gray_edges))


def detect_lane_segments(frame):
    """Retorna segmentos cuja extensao coincide com uma lateral da pista."""
    height, width = frame.shape[:2]
    y_top = int(height * 0.46)
    y_bottom = int(height * 0.68)
    lines = cv2.HoughLinesP(
        build_lane_edges(frame),
        rho=1,
        theta=np.pi / 180,
        threshold=12,
        minLineLength=max(14, int(width * 0.035)),
        maxLineGap=max(40, int(width * 0.10)),
    )

    accepted = []
    if lines is None:
        return accepted

    for line in lines:
        x1, y1, x2, y2 = line[0]
        dx = x2 - x1
        if abs(dx) < 2:
            continue

        slope = (y2 - y1) / dx
        if not 0.18 <= abs(slope) <= 1.8:
            continue

        intercept = y1 - slope * x1
        x_top = (y_top - intercept) / slope
        x_bottom = (y_bottom - intercept) / slope

        is_left = (
            slope < 0
            and x_bottom < width * 0.48
            and width * 0.20 < x_top < width * 0.55
            and x_bottom < x_top
        )
        is_right = (
            slope > 0
            and x_bottom > width * 0.52
            and width * 0.45 < x_top < width * 0.80
            and x_top < x_bottom
        )
        if is_left or is_right:
            accepted.append(line)

    return accepted

In [28]:
# Gerar vídeo apenas com ROI aplicado (referência visual)
process_video(apply_roi, "01_roi")

✓ Vídeo salvo: outputs/01_roi.mp4 (1674 frames)


### Etapa 2: Pré-processamento (com ROI aplicado)

In [29]:
def color_space(frame):
    """Conversão para espaço de cor HSV (com ROI aplicado)"""
    frame = apply_roi(frame)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    return hsv

process_video(color_space, "02_hsv")

✓ Vídeo salvo: outputs/02_hsv.mp4 (1674 frames)


### Máscaras de Cor + Limiarização

In [ ]:
def color_mask(frame):
    """Mascara de pintura branca/cinza-clara e amarela dentro da pista."""
    return cv2.cvtColor(build_color_mask(frame), cv2.COLOR_GRAY2BGR)


process_video(color_mask, "03_mask")

### Filtro Gaussiano

In [ ]:
def gaussian_blur(frame):
    """Suaviza a mascara de cor usada pelo detector."""
    blur = cv2.GaussianBlur(build_color_mask(frame), (5, 5), 0)
    return cv2.cvtColor(blur, cv2.COLOR_GRAY2BGR)


process_video(gaussian_blur, "04_blur")

### Detecção de Bordas (Canny)

In [ ]:
def edges(frame):
    """Bordas de cor e de contraste, limitadas ao trapezio da pista."""
    return cv2.cvtColor(build_lane_edges(frame), cv2.COLOR_GRAY2BGR)


process_video(edges, "05_edges")

### Transformada de Hough

In [ ]:
def hough_lines(frame):
    """Mostra somente segmentos Hough coerentes com as bordas da pista."""
    line_img = np.zeros_like(frame)
    for line in detect_lane_segments(frame):
        x1, y1, x2, y2 = line[0]
        cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 4)

    return cv2.addWeighted(frame, 0.85, line_img, 1, 0)


process_video(hough_lines, "06_hough")

### Rastreamento das Faixas (Suavização)

In [ ]:
# ==================== RASTREAMENTO DE FAIXAS ====================
def reset_lane_state():
    """Limpa o historico antes de processar um novo video."""
    global _lane_state
    _lane_state = {
        "left": None, "right": None,
        "left_history": [], "right_history": [],
        "left_misses": 0, "right_misses": 0,
    }


def average_slope_intercept(lines, frame):
    """Media ponderada por comprimento para cada lateral da pista."""
    groups = {"left": [], "right": []}
    for line in lines:
        x1, y1, x2, y2 = line[0]
        dx = x2 - x1
        if abs(dx) < 2:
            continue
        slope = (y2 - y1) / dx
        intercept = y1 - slope * x1
        length = np.hypot(dx, y2 - y1)
        groups["left" if slope < 0 else "right"].append(
            (slope, intercept, length)
        )

    result = {}
    for side, values in groups.items():
        if values:
            values = np.asarray(values, dtype=float)
            result[side] = np.average(
                values[:, :2], axis=0, weights=values[:, 2]
            )
        else:
            result[side] = None
    return result["left"], result["right"]


def make_line_points(frame, line):
    """Converte inclinacao/intercepto nos quatro pontos usados no desenho."""
    height, width = frame.shape[:2]
    slope, intercept = line
    y_bottom = int(height * 0.68)
    y_top = int(height * 0.46)
    x_bottom = np.clip((y_bottom - intercept) / slope, 0, width - 1)
    x_top = np.clip((y_top - intercept) / slope, 0, width - 1)
    return np.array([x_bottom, y_bottom, x_top, y_top], dtype=float)


def update_lane_state(frame):
    """Estabiliza pontos com mediana, rejeicao de saltos e EMA lenta."""
    height, width = frame.shape[:2]
    left, right = average_slope_intercept(detect_lane_segments(frame), frame)

    for side, measured_line in (("left", left), ("right", right)):
        misses_key = f"{side}_misses"
        history_key = f"{side}_history"

        if measured_line is None:
            _lane_state[misses_key] += 1
            if _lane_state[misses_key] > 25:
                _lane_state[side] = None
                _lane_state[history_key].clear()
            continue

        measured = make_line_points(frame, measured_line)
        previous = _lane_state[side]

        # Ignora uma medicao isolada que desloca demais a faixa.
        if previous is not None:
            displacement = np.max(np.abs(measured[[0, 2]] - previous[[0, 2]]))
            if displacement > width * 0.14:
                _lane_state[misses_key] += 1
                continue

        _lane_state[misses_key] = 0
        history = _lane_state[history_key]
        history.append(measured)
        if len(history) > 9:
            history.pop(0)

        median_points = np.median(np.asarray(history), axis=0)
        if previous is None:
            _lane_state[side] = median_points
        else:
            alpha = 0.10
            _lane_state[side] = (1 - alpha) * previous + alpha * median_points

    return _lane_state["left"], _lane_state["right"]


def valid_lane_pair(frame, left, right):
    """Confere se as linhas formam uma faixa sem cruzar."""
    if left is None or right is None:
        return False
    width = frame.shape[1]
    bottom_width = right[0] - left[0]
    top_width = right[2] - left[2]
    return (
        width * 0.30 < bottom_width < width * 1.05
        and width * 0.08 < top_width < width * 0.65
    )


def draw_tracked_lines(frame, left, right):
    """Desenha as faixas rastreadas em vermelho."""
    line_img = np.zeros_like(frame)
    for points in (left, right):
        if points is not None:
            x1, y1, x2, y2 = np.rint(points).astype(int)
            cv2.line(line_img, (x1, y1), (x2, y2), (0, 0, 255), 6)
    return cv2.addWeighted(frame, 0.88, line_img, 1, 0)


def track_lanes(frame):
    """Rastreia e desenha as duas bordas estabilizadas da pista."""
    left, right = update_lane_state(frame)
    return draw_tracked_lines(frame, left, right)


reset_lane_state()
process_video(track_lanes, "07_tracking")

## Etapa 3: Área da faixa em que o carro está

O vídeo final preenche em verde a região entre as duas linhas rastreadas.

In [ ]:
def lane_area(frame):
    """Destaca em verde a faixa estimada em que o carro esta."""
    left, right = update_lane_state(frame)
    if not valid_lane_pair(frame, left, right):
        return draw_tracked_lines(frame, left, right)

    lx_bottom, y_bottom, lx_top, y_top = np.rint(left).astype(int)
    rx_bottom, _, rx_top, _ = np.rint(right).astype(int)
    polygon = np.array([[
        (lx_bottom, y_bottom),
        (lx_top, y_top),
        (rx_top, y_top),
        (rx_bottom, y_bottom),
    ]], dtype=np.int32)

    overlay = np.zeros_like(frame)
    cv2.fillPoly(overlay, polygon, (0, 255, 0))
    cv2.polylines(overlay, polygon, True, (255, 0, 255), 4)
    return cv2.addWeighted(frame, 1.0, overlay, 0.30, 0)


reset_lane_state()
process_video(lane_area, "08_lane_area")